# MiningMedicinalTaxa (Example Notebook)
Extract plant names, medical conditions, and medicinal effects from text using **SciBERT** (NER + RE) and **GPT** (structured extraction).

Both methods return the same `TaxaData` schema so results are directly comparable.

In [ ]:
# Install
# Run once, then RESTART RUNTIME
!pip install -q git+https://github.com/alrichardbollans/wcvpy.git
!pip install -q git+https://github.com/alrichardbollans/MiningMedicinalTaxa.git
!pip install -q "torchao>=0.16.0"

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [ ]:
#Get the sample text file, or upload your own.
import os, urllib.request

# Clear any previously-uploaded .txt files to not accidentally process old input
for f in [f for f in os.listdir('.') if f.endswith('.txt')]:
    os.remove(f)
    print(f'Removed previous upload: {f}')

SAMPLE_URL = "https://raw.githubusercontent.com/alrichardbollans/MiningMedicinalTaxa/main/R/sample.txt"
txt_name = "sample.txt"
urllib.request.urlretrieve(SAMPLE_URL, txt_name)

# To use your own file instead, comment out the line above and uncomment below:
# from google.colab import files
# uploaded = files.upload()
# txt_name = list(uploaded.keys())[0]

print(f"Using: {txt_name}")

Using: sample.txt


In [ ]:
# set API key (for GPT extraction only)

# Load OpenAI key from Colab secrets (add your apy key clicking the icon in left sidebar)
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# Pretty-print helper. It works for both SciBERT and GPT output
def print_taxa(taxa_data, title='Results'):
    taxa = taxa_data.taxa or []
    print(f'\n{"=" * 60}')
    print(f'  {title} — {len(taxa)} taxa found')
    print(f'{"=" * 60}')
    for i, taxon in enumerate(taxa, 1):
        print(f'\n  [{i}] {taxon.scientific_name}')
        conditions = taxon.medical_conditions
        effects = taxon.medicinal_effects
        if conditions:
            print(f'      Conditions: {", ".join(str(c) for c in conditions) if isinstance(conditions, list) else conditions}')
        else:
            print(f'      Conditions: —')
        if effects:
            print(f'      Effects:    {", ".join(str(e) for e in effects) if isinstance(effects, list) else effects}')
        else:
            print(f'      Effects:    —')
    print(f'\n{"=" * 60}\n')

## 1. SciBERT Extraction
Fine-tuned SciBERT NER and RE models. Runs locally, no API key needed.

In [ ]:
# Load SciBert models
from SciBert.running_scibert import load_scibert
models = load_scibert()

In [ ]:
# Extraction
# run_re=True also extracts relations (slower)

from SciBert.running_scibert import query_scibert
from LLM_models.evaluating import clean_model_annotations_using_taxonomy_knowledge
import json

scibert_output = query_scibert(models, txt_name, json_dump='scibert_output.json', run_re=True)
# optional: use a taxonomy filter to remove vernacular names
scibert_output = clean_model_annotations_using_taxonomy_knowledge(scibert_output)

print_taxa(scibert_output, 'SciBERT')

## 2. GPT Extraction


In [ ]:
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from LLM_models.running_models import query_a_model, get_input_size_limit
from LLM_models.evaluating import clean_model_annotations_using_taxonomy_knowledge

# Specify a .env file containing your API key in the form OPENAI_API_KEY="key",
# or alternatively specify the apikey directly with the apikey parameter
load_dotenv(dotenv_path='.env')

# Set up a GPT model
gpt_model = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0)
context_window = get_input_size_limit(5)  # 5k tokens per chunk (max 128k tokens)

gpt_output = query_a_model(gpt_model, txt_name,
                              context_window, json_dump='gpt_output.json', single_chunk=False)

# If you want to clean outputs by removing annotations with unknown scientific names:
gpt_output = clean_model_annotations_using_taxonomy_knowledge(gpt_output)

print_taxa(gpt_output, 'GPT-4o')

Outputs from this process (the json_dump files) can be manually verified using our reference verifier shiny app, hosted here: __https://huggingface.co/spaces/alrichardbollans/MedicinalTaxonVerifier__

In [ ]:
files.download('scibert_output.json')
files.download('gpt_output.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>